# mart_dev.run_log

A Saved table made by the batch run_log, with project.convert(...), from the statements in `conversion/statement.hql`, as its script wrote them. Its logic cell builds the same query from the Toolbox's own functions, with `create()`, `preview()` and `write()`.

## Terms

- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Logic cell**: the code cell whose first line is `# logic`: the table's imports and functions alone.
- **Decision**: one thing about this table that only you decide, such as its location or its Key; `conversion/checklist.md` lists each one still open.
- **Key**: the columns whose values name exactly one row of the table.
- **Original**: the table the converted script writes, in the warehouse already, which this table copies, or, for a table made from a query, that query. `conversion/in_dev.hql` holds the original's Hive, which the cells comparing it read.

## Steps

1. **Answer its Decisions.** Goal: each Decision made. Action: answer each open Decision `conversion/checklist.md` lists in the decisions cell of `admin/workflows/convert/convert.ipynb`. Done when `conversion/checklist.md` says None under Open Decisions, once that Workflow's next step writes your answers in; then step 2.
2. **Run the cells.** Goal: its rows, its Hive and its lineage, tried. Action: run the code cells top to bottom. Done when each runs with no error.

## Decision rules

- If you still want to compare it with the original, keep the cells comparing them.
- If you don't, delete those cells: they stay until you delete them.

## Self-check

- `conversion/checklist.md` says None under Open Decisions.
- Each code cell runs top to bottom with no error.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project: two up

from sqlglot_composer import (compare_hive, compare_with_original, export_hive, export_lineage,
                              project, run_query, show_hive)
from warehouse import query

In [ ]:
# logic
from sqlglot_composer import (
    SessionSettings,
    Statement,
    hive_settings,
    unchecked_hive,
)
from sqlglot_composer import tag_text  # for each Script tag in a string
from sqlglot_composer import create_table  # for create()
from sqlglot_composer import UncheckedStatement, unchecked_hive  # for write()
from references.mart_dev__run_log import run_log, TAGS, SEED  # its Table reference, and its Script tags
from sqlglot_composer import UncheckedStatement, seed_copy  # for seed()

HIVE_SETTINGS = {"converted": {
    "hive.exec.dynamic.partition.mode": "nonstrict",
}}


def session_settings() -> SessionSettings:
    """The SET lines the original sent before its query: project.run_tables
    sends them before the write."""
    return hive_settings("converted", sets=HIVE_SETTINGS)


def converted() -> Statement:
    """The query, as the Hive this Conversion was made from writes it."""
    return unchecked_hive("SELECT r.run_id, r.job_id, r.dt FROM ops.job_runs AS r WHERE r.dt BETWEEN DATE_ADD(" + tag_text(TAGS["PERIOD_END"], quoted=False) + ", 1 * -1) AND " + tag_text(TAGS["PERIOD_END"], quoted=False) + " AND NOT r.run_id IN (SELECT l.run_id FROM mart_dev.run_log AS l)")


def create() -> Statement:
    """Create mart_dev.run_log from its Table reference, if it isn't there yet."""
    return create_table(run_log, may_exist=True)


def preview() -> Statement:
    """The rows write() would save, as a SELECT to run and check first."""
    return converted()


def write() -> UncheckedStatement:
    """Save the rows into mart_dev.run_log, as the original's INSERT did: kept as Hive,
    as converted() is.

    project.run_tables sends session_settings() first, as the original sent its
    SET lines.
    """
    return unchecked_hive("INSERT INTO mart_dev.run_log PARTITION(dt) SELECT r.run_id, r.job_id, r.dt FROM ops.job_runs AS r WHERE r.dt BETWEEN DATE_ADD(" + tag_text(TAGS["PERIOD_END"], quoted=False) + ", 1 * -1) AND " + tag_text(TAGS["PERIOD_END"], quoted=False) + " AND NOT r.run_id IN (SELECT l.run_id FROM mart_dev.run_log AS l)")


def seed() -> UncheckedStatement:
    """Copy the original's rows into this table once, before its first write():
    its write reads its own table, which starts empty. SEED, in its Table reference,
    limits the Partitions it copies."""
    return seed_copy(run_log, "mart.run_log", SEED)


In [ ]:
# Try it: its rows, then, once its Table reference has its location=, the Hive of
# each step
run_query(preview(), query)
show_hive([create(), write()])

In [ ]:
# Lineage: a new pair in lineage/, named by the time and this table
export_lineage(write())

In [ ]:
# Its Hive: mart_dev__run_log.hql beside this notebook, from the logic as last saved
export_hive()

In [ ]:
# Its Hive against the original's, as conversion/in_dev.hql holds it
ORIGINAL = pathlib.Path("conversion/in_dev.hql").read_text(encoding="utf-8")
show_hive(preview())
print(compare_hive(preview(), ORIGINAL))

In [ ]:
# The convert Workflow's compare step, project.compare_batch, compares this table once
# written. This compares its query with the original's first:
# Its rows against the original's, through your query, on one Partition:
# print(compare_with_original(preview(), ORIGINAL, query=query, partition={"dt": "<a value>"}))

In [ ]:
# Its rows in the Reference sheet
project.refresh_reference_sheet("mart_dev.run_log")